# Train a model, save it, use it (version 2: three predictors)

Version 1 used **temperature** only. In this version I add two more things that affect ice cream sales:

- **is_weekend**: more people are out on Saturdays and Sundays
- **precipitation**: rain (mm) keeps people at home

Both can be filled in automatically by the app: the weekend flag comes from the date, and precipitation comes from the same weather API as the temperature.

The data is **made up**, to show the steps.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.linear_model import LinearRegression
import pickle

## Made-up data

Four hundred days (more than version 1, so the model can see rain and weekends). Hotter days sell more, weekends sell more, rainy days sell less.

In [2]:
np.random.seed(42)
n = 400

temperature = np.random.uniform(50, 100, n)                        # daily high in Fahrenheit
is_weekend = np.random.randint(0, 2, n)                            # 0 = weekday, 1 = weekend
rainy = np.random.rand(n) < 0.3                                    # about 30% of days have rain
precipitation = np.where(rainy, np.random.uniform(1, 25, n), 0.0)  # mm of rain

sales = 5 * temperature - 100 + 40 * is_weekend - 3 * precipitation + np.random.normal(0, 25, n)

df = pd.DataFrame({"temperature": temperature, "is_weekend": is_weekend,
                   "precipitation": precipitation, "sales": sales.clip(0)})
df.head()

,temperature,is_weekend,precipitation,sales
0,68.727006,0,5.443190,218.589155
1,97.535715,1,14.005623,376.930266
2,86.599697,0,21.950700,259.105509
3,79.932924,0,18.573397,295.863129
4,57.800932,1,20.357468,177.480644


In [3]:
px.scatter(df, x="temperature", y="sales", color="is_weekend", title="Cones sold versus daily high (1 = weekend)").show()
px.scatter(df, x="precipitation", y="sales", title="Cones sold versus rain (mm)").show()

## Fit the model

The model now has three inputs. The column names and their order must match what the app sends.

In [4]:
FEATURES = ["temperature", "is_weekend", "precipitation"]
x = df[FEATURES]
y = df["sales"]

model = LinearRegression()
model.fit(x, y)

print("Intercept:", round(model.intercept_, 1))
for name, coef in zip(FEATURES, model.coef_):
    print(f"{name}: {coef:.2f}")
print("R2:", round(model.score(x, y), 2))

Intercept: -86.9
temperature: 4.88
is_weekend: 37.18
precipitation: -3.27
R2: 0.91


Each extra degree adds about five cones, a weekend adds about 40, and each mm of rain removes about 3. Now a prediction for an 85 °F Saturday with no rain.

In [5]:
new_day = pd.DataFrame({"temperature": [85], "is_weekend": [1], "precipitation": [0.0]})
print(round(model.predict(new_day)[0]))

365


## Save it

In [6]:
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

## Load it back and check that it works

In [7]:
with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

print(round(loaded_model.predict(new_day)[0]))   # the same number as before

365


## Download it

The file has to go in the GitHub repository, next to `app.py`. (This cell only works in Google Colab.)

In [8]:
try:
    from google.colab import files
    files.download("model.pkl")
except ImportError:
    print("Not in Colab: model.pkl is saved in the current folder.")

Not in Colab: model.pkl is saved in the current folder.
